In [ ]:
# Generate Spectus database metrics for 2000 users
from datetime import date
import pandas as pd
from pathlib import Path
from src.map_terminology import map_to_shared_mode_names
from src.optimized_analysis import get_prediction_for_trip_rmove
from src.calculate_speed import calculate_speed_for_dataframe, haversine

today = date.today().strftime("%Y%m%d")

root_path = 'data/Spectus/Lyra_Processed/'
input_path = f'{root_path}split_by_user/'
output_file_path = f'{root_path}{today}_downsampling_masks.csv'

user_count = 0
total_trips_count = 0
for file in Path(input_path).iterdir():
    if not file.is_file():
        continue

    trip_count = 0
    try:
        print(f'Processing user {user_count}')
        locations_df = pd.read_csv(f'{input_path}{file.name}', low_memory=False)

        # Remove stop points
        locations_df = locations_df[locations_df['traj_id'] != -99]

        locations_df['traj_id'] = locations_df['user_ID'].astype(str) + '_' + locations_df['traj_id'].astype(str)

        rows = []
        for traj_id, df in locations_df.groupby('traj_id'):

            df = df.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
            df['timestamp'] = pd.to_datetime(df['timestamp'])
            df = df.sort_values('timestamp')

            datapoints_bucketed_by_5s = df.set_index('timestamp').resample('5s').size()
            start_time = datapoints_bucketed_by_5s.index[0]
            for i, (timestamp, record_count) in enumerate(datapoints_bucketed_by_5s.items()):
                if record_count == 0:
                    continue

                bin_start_elapsed_sec = (timestamp - start_time).total_seconds()
                bin_size_sec = 5.0
                bin_end_elapsed_sec = bin_start_elapsed_sec + bin_size_sec

                rows.append({
                    'trip_id': traj_id,
                    'bin_size_sec': bin_size_sec,
                    'bin_index': i,
                    'bin_start_elapsed_sec': bin_start_elapsed_sec,
                    'bin_end_elapsed_sec': bin_end_elapsed_sec,
                    'record_count': record_count
                })

            datapoints_bucketed_by_20s = df.set_index('timestamp').resample('20s').size()
            start_time = datapoints_bucketed_by_20s.index[0]
            for i, (timestamp, record_count) in enumerate(datapoints_bucketed_by_20s.items()):
                if record_count == 0:
                    continue

                bin_start_elapsed_sec = (timestamp - start_time).total_seconds()
                bin_size_sec = 20.0
                bin_end_elapsed_sec = bin_start_elapsed_sec + bin_size_sec

                rows.append({
                    'trip_id': traj_id,
                    'bin_size_sec': bin_size_sec,
                    'bin_index': i,
                    'bin_start_elapsed_sec': bin_start_elapsed_sec,
                    'bin_end_elapsed_sec': bin_end_elapsed_sec,
                    'record_count': record_count
                })

            datapoints_bucketed_by_30s = df.set_index('timestamp').resample('30s').size()
            start_time = datapoints_bucketed_by_30s.index[0]
            for i, (timestamp, record_count) in enumerate(datapoints_bucketed_by_30s.items()):
                if record_count == 0:
                    continue

                bin_start_elapsed_sec = (timestamp - start_time).total_seconds()
                bin_size_sec = 30.0
                bin_end_elapsed_sec = bin_start_elapsed_sec + bin_size_sec

                rows.append({
                    'trip_id': traj_id,
                    'bin_size_sec': bin_size_sec,
                    'bin_index': i,
                    'bin_start_elapsed_sec': bin_start_elapsed_sec,
                    'bin_end_elapsed_sec': bin_end_elapsed_sec,
                    'record_count': record_count
                })

            datapoints_bucketed_by_60s = df.set_index('timestamp').resample('1min').size()
            start_time = datapoints_bucketed_by_60s.index[0]
            for i, (timestamp, record_count) in enumerate(datapoints_bucketed_by_60s.items()):
                if record_count == 0:
                    continue

                bin_start_elapsed_sec = (timestamp - start_time).total_seconds()
                bin_size_sec = 60.0
                bin_end_elapsed_sec = bin_start_elapsed_sec + bin_size_sec

                rows.append({
                    'trip_id': traj_id,
                    'bin_size_sec': bin_size_sec,
                    'bin_index': i,
                    'bin_start_elapsed_sec': bin_start_elapsed_sec,
                    'bin_end_elapsed_sec': bin_end_elapsed_sec,
                    'record_count': record_count
                })

            trip_count += 1

        dataframe = pd.DataFrame(rows)
        dataframe = dataframe.set_index('trip_id')
        if user_count == 0:
            dataframe.to_csv(output_file_path, index=True, header=True, mode='w')
        else:
            dataframe.to_csv(output_file_path, index=True, header=False, mode='a')
    except Exception as e:
        print(f"Unexpected error: {e}")
        print(f"file: {file.name}")

    user_count += 1
    print(f'Appended {trip_count} trips')
    total_trips_count += trip_count

print("Done!")
print(f"Total trips processed: {total_trips_count}")